In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
# Task 1: Write your code here:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns="Order_ID", axis=1)
df.head()


In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
cols = ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs', 'Delivery_Time']
df_clean = df[cols].copy()

# Drop rows where target  missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df_clean.dropna(subset=['Delivery_Time'])
print(f"After dropping missing : {df_clean.shape}")

In [ ]:
check_missing_values(df_clean)

In [ ]:
# # Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day''Time_of_Day', 'Vehicle_Type']:
     df_clean[col] = df_clean[col].fillna('unknown')

#Fill Courier_Experience_yrs with mean
df_clean['Courier_Experience_yrs'] = df_clean['Courier_Experience_yrs'].fillna(df_clean['Courier_Experience_yrs'].mean())

print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
# 4. Do we have duplicate samples?
def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

catog = ['Weather',  'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
#categories = pd.DataFrame(catog)
catog


In [ ]:
from sklearn.preprocessing import LabelEncoder #import LabelEncoder

print('data before encoding:\n', catog) #show before encoding

label_encoder = LabelEncoder() # Instantiate LabelEncoder
data_label_encoded = label_encoder.fit_transform(catog) # Apply fit_transform to the column

print('\nData after encoding:\n', data_label_encoded) #show after encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
# 3. Do we have categorical columns?
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))
label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:

# onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
# data_onehot_encoded = onehot_encoder.fit_transform(cols) # Apply fit_transform to the copied

# print('\nData after encoding:\n', data_onehot_encoded) #show after encoding

In [ ]:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  #we shouldnt SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [ ]:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split

n_splits = 3 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

In [ ]:
# Train Random Forest Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")


In [ ]:
print("Linear Regression Results")
print(f"  Average MSE: {np.mean(mae):.4f}")

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['Lasso'] = model.coef_
coeffs['Ridge'] = model.coef_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Plot for Linear Regression Predictions vs. Ground Truth
plt.figure(figsize=(6, 4))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], 'r--', linewidth=2)
plt.xlabel("Actual y_test (Ground Truth)")
plt.ylabel("Predicted y_pred (Linear Regression)")
plt.title("Predictions vs. Ground Truth")
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Task Bonus: Write your code here: